In [2]:
import akshare as ak
import baostock as bs
import pandas as pd
import os
import time,random
import re

In [3]:
# 1. 设置保存路径 (你可以改成自己需要的路径)
# save_dir = r"C:\data\日K线"
save_dir = r"C:\data\退市票日K线"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 2. 自由设置你需要下载的时间范围
START_DATE = '2010-01-01'
END_DATE = '2026-3-21'
# adjustflag: "1"是后复权, "2"是前复权, "3"是不复权。(注意Baostock的复权参数跟新浪不同，这里用3代表不复权)
ADJUST_FLAG = "3"
#
# # 3. 使用 akshare 获取最新、最全的A股股票代码表
# print("正在通过 akshare 获取A股所有股票代码列表...")
# stock_info_df = ak.stock_info_a_code_name()
#退市股票
# 获取沪市退市名单
df_sh_delist = ak.stock_info_sh_delist()
df_sh_delist = df_sh_delist[['公司代码', '公司简称']]
df_sh_delist.columns = ['code', 'name'] # 统一列名
# 获取深市退市名单
df_sz_delist = ak.stock_info_sz_delist()
df_sz_delist = df_sz_delist[['证券代码', '证券简称']]
df_sz_delist.columns = ['code', 'name'] # 统一列名
# 合并沪深退市名单
stock_info_df = pd.concat([df_sh_delist, df_sz_delist], ignore_index=True)



total_stocks = len(stock_info_df)
print(f"共获取到 {total_stocks} 只股票，准备开始批量下载...\n")

共获取到 350 只股票，准备开始批量下载...



In [4]:


def download_all_baostock_k_data():


    # 4. 登录 baostock 系统 (只需要在循环开始前登录一次即可)
    print("正在登录 Baostock 系统...")
    lg = bs.login()
    if lg.error_code != '0':
        print(f"Baostock 登录失败: {lg.error_msg}")
        return
    print("Baostock 登录成功！\n")

    # 5. 遍历每一只股票
    for index, row in stock_info_df.iterrows():
        base_code = str(row['code']).zfill(6)
        stock_name = row['name']

        # 转换成 Baostock 需要的格式 (注意这里中间加上了 ".")
        if base_code.startswith(('60', '68', '69')):
            bs_symbol = f"sh.{base_code}"
        elif base_code.startswith(('00', '30')):
            bs_symbol = f"sz.{base_code}"
        elif base_code.startswith(('4', '8', '9')):
            bs_symbol = f"bj.{base_code}"
        else:
            continue

        # 清洗文件名，防止 *ST 股票带有星号导致 Windows 报错
        safe_stock_name = re.sub(r'[\\/:*?"<>|]', '', stock_name)
        file_path = os.path.join(save_dir, f"{bs_symbol}_{safe_stock_name}_日K.csv")

        # 断点续传逻辑：文件存在则瞬间跳过
        if os.path.exists(file_path):
            print(f"[{index + 1}/{total_stocks}] {bs_symbol} {stock_name} 已存在，跳过...")
            continue

        try:
            # 6. 调用 Baostock 接口获取数据
            rs = bs.query_history_k_data_plus(
                bs_symbol,
                "date,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,peTTM,pbMRQ,psTTM,pcfNcfTTM,isST",
                start_date=START_DATE,
                end_date=END_DATE,
                frequency="d",
                adjustflag=ADJUST_FLAG
            )

            # 判断接口调用是否成功
            if rs.error_code != '0':
                print(f"[{index + 1}/{total_stocks}] 下载失败 {bs_symbol} {stock_name} 接口返回错误: {rs.error_msg}")
                continue

            # 7. 将结果集装入 list 并转换为 DataFrame
            data_list =[]
            while (rs.error_code == '0') and rs.next():
                data_list.append(rs.get_row_data())

            result_df = pd.DataFrame(data_list, columns=rs.fields)

            # 8. 保存到 CSV
            # 判断是否有数据 (部分新股或停牌股可能在这段时间内没有任何数据)
            if not result_df.empty:
                result_df.to_csv(file_path, index=False, encoding='gbk')
                print(f"[{index + 1}/{total_stocks}] 下载成功: {bs_symbol} {stock_name}")
            else:
                print(f"[{index + 1}/{total_stocks}] 数据为空: {bs_symbol} {stock_name} (可能是该区间无数据/未上市)")

            # Baostock 是官方提供的高性能API，对并发和频率容忍度非常高。
            # 这里只需设置微小休眠(0.1秒)防止本地网络TCP连接数耗尽即可，下载速度会比爬虫快几十倍！
            time.sleep(random.uniform(0, 1))

        except Exception as e:
            print(f"[{index + 1}/{total_stocks}] {bs_symbol} {stock_name} 发生异常: {e}")

    # 9. 所有循环结束后，一定要登出 Baostock
    bs.logout()
    print("\nBaostock 登出成功，全部任务执行完毕！")

if __name__ == "__main__":

    download_all_baostock_k_data()

正在登录 Baostock 系统...
login success!
Baostock 登录成功！

[1/350] 数据为空: sh.600001 邯郸钢铁 (可能是该区间无数据/未上市)
[2/350] 数据为空: sh.600002 齐鲁退市 (可能是该区间无数据/未上市)
[3/350] sh.600003 ST东北高 已存在，跳过...
[4/350] sh.600005 武钢股份 已存在，跳过...
[5/350] 数据为空: sh.600065 *ST联 谊 (可能是该区间无数据/未上市)
[6/350] sh.600068 葛洲坝 已存在，跳过...
[7/350] sh.600069 退市银鸽 已存在，跳过...
[8/350] sh.600070 *ST富润 已存在，跳过...
[9/350] sh.600074 退市保千 已存在，跳过...
[10/350] sh.600077 *ST宋都 已存在，跳过...
[11/350] sh.600083 *ST博信 已存在，跳过...
[12/350] sh.600086 退市金钰 已存在，跳过...
[13/350] sh.600087 退市长油 已存在，跳过...
[14/350] sh.600090 退市济堂 已存在，跳过...
[15/350] sh.600091 退市明科 已存在，跳过...
[16/350] 数据为空: sh.600092 *ST精 密 (可能是该区间无数据/未上市)
[17/350] sh.600093 退市易见 已存在，跳过...
[18/350] sh.600102 莱钢股份 已存在，跳过...
[19/350] sh.600112 *ST天成 已存在，跳过...
[20/350] sh.600122 *ST宏图 已存在，跳过...
[21/350] sh.600139 *ST西源 已存在，跳过...
[22/350] sh.600145 退市新亿 已存在，跳过...
[23/350] sh.600146 退市环球 已存在，跳过...
[24/350] sh.600175 退市美都 已存在，跳过...
[25/350] 数据为空: sh.600181 *ST云 大 (可能是该区间无数据/未上市)
[26/350] sh.600190 退市锦港 已存在，跳过...
[2